# 05_get_sequences_align_and_analyse_conservation

Scaffold notebook for sequence/structure alignment, MSA visualization, and optional conservation analysis.


## Python Path Setup


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
src_root = repo_root / "src"
if src_root.exists() and str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))


## Imports


In [ ]:
import importlib
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

import agentic_protein_design.steps.get_sequences_align_and_analyse_conservation as align_step
align_step = importlib.reload(align_step)

default_user_inputs = align_step.default_user_inputs
run_alignment_and_conservation = align_step.run_alignment_and_conservation


## Workflow Skeleton

1. Build an MSA from sequence or structure inputs.
2. Plot and save an alignment visualization.
3. Optionally compute per-position conservation metrics.


## User Inputs (Edit This Cell)


In [ ]:
user_inputs = default_user_inputs()

# Data root key from project_config/variables.py (address_dict)
user_inputs["root_key"] = "examples"
user_inputs["data_subfolder"] = ""

# Alignment input mode
# - "sequence": align input sequences directly
# - "structure": extract sequences from PDBs, then align
user_inputs["alignment_input_mode"] = "sequence"

# Sequence backend
# - "openprotein": homolog-aware MSA from seed sequence (API)
# - "mafft": local MAFFT using provided sequences
user_inputs["sequence_alignment_backend"] = "mafft"

# If OpenProtein is selected, provide seed_sequence and/or input_sequences
user_inputs["seed_sequence"] = ""

# Inline sequence list (optional)
user_inputs["input_sequences"] = [
    # "MSEQUENCEEXAMPLE",
]

# FASTA files under {root}/{sequence_subdirectory}/{data_subfolder}/
user_inputs["sequence_subdirectory"] = "sequences/"
user_inputs["sequence_fasta_filenames"] = [
    # "homologs.fasta",
]

# PDB files under {root}/{structure_subdirectory}/{data_subfolder}/
# Used when alignment_input_mode = "structure"
user_inputs["structure_subdirectory"] = "pdb/"
user_inputs["structure_pdb_filenames"] = [
    # "ET096.pdb",
]

# Local MAFFT binary path or command name
user_inputs["mafft_executable"] = "mafft"

# Output file names (saved under processed/05_get_sequences_align_and_analyse_conservation/)
user_inputs["msa_output_filename"] = "msa_aligned.fasta"
user_inputs["plot_output_filename"] = "msa_visualization.png"
user_inputs["conservation_output_filename"] = "msa_conservation.csv"

# Visualization / conservation options
user_inputs["max_sequences_for_plot"] = 120
user_inputs["max_positions_for_plot"] = 600
user_inputs["run_conservation_analysis"] = True

user_inputs


## Run Alignment Pipeline


In [ ]:
result = run_alignment_and_conservation(user_inputs)
result["status"], result["alignment_input_mode"], result["alignment_backend"]


## Output Paths


In [ ]:
summary = {
    "data_root": result.get("data_root", ""),
    "processed_dir": result.get("processed_dir", ""),
    "msa_path": result.get("msa_path", ""),
    "plot_path": result.get("plot_path", ""),
    "conservation_path": result.get("conservation_path", ""),
    "n_input_sequences": result.get("n_input_sequences", 0),
}
pd.Series(summary)


## MSA Preview


In [ ]:
msa_path = Path(result["msa_path"])
if msa_path.exists():
    print("MSA file:", msa_path)
    print("-" * 80)
    print("\n".join(msa_path.read_text(encoding="utf-8").splitlines()[:30]))
else:
    print("MSA file not found.")


## MSA Plot


In [ ]:
plot_path = Path(result["plot_path"])
if plot_path.exists():
    print("MSA plot:", plot_path)
    display(Image(filename=str(plot_path)))
else:
    print("MSA plot not found.")


## Conservation Analysis (Optional)


In [ ]:
cons_path = Path(result.get("conservation_path", ""))
if cons_path and cons_path.exists():
    cons_df = pd.read_csv(cons_path)
    print("Conservation file:", cons_path)
    cons_df.head(20)
else:
    print("Conservation analysis skipped or no output generated.")
